# Clase 2 — Filtrado, convolución, ruido y bordes

**Unidad VII · 4 h · grupos de 3**

> **La pregunta de hoy:** ¿cómo limpio una imagen y cómo encuentro su estructura?

## 1. Objetivos

Al terminar este cuaderno tienes que poder:

1. Explicar qué es un kernel y qué hace la convolución, y **predecir** el efecto de un kernel antes de aplicarlo.
2. Distinguir tres modelos de ruido —gaussiano, sal y pimienta— y elegir filtro **midiendo**, no mirando.
3. Calcular el gradiente de una imagen con Sobel y explicar por qué las derivadas encuentran bordes.
4. Recorrer las cuatro etapas de Canny y **romperlo a propósito** controlando un umbral.
5. Leer una tabla de tiempos y decidir si un filtro cabe en un presupuesto de 16,6 ms.

Este cuaderno se ejecuta de arriba abajo. Si algo falla, para y arréglalo antes de seguir.

## Preparación

Funciona en dos sitios:

* **En tu máquina**, con el repositorio del motor clonado: el apartado de Canny etapa por etapa usa la implementación didáctica de `edge_detection.py`.
* **En Google Colab**, sin nada: los experimentos usan NumPy y OpenCV, que Colab trae, y el cuaderno sigue siendo ejecutable de principio a fin.

No hay celdas con `!pip install` ni `!git clone` a propósito: se comprueba y se avisa, en Python, para que la misma celda valga en los dos sitios.

In [ ]:
import os
import sys
from pathlib import Path

# Sin pantalla: el motor arranca en modo headless. Hay que fijarlo ANTES de
# importar pygame, porque SDL elige el controlador de vídeo al inicializarse.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")


def preparar_rutas() -> bool:
    """Pone `cvcourse` en el path. Devuelve si además hay motor."""
    for candidato in (Path.cwd(), *Path.cwd().parents):
        if (candidato / "cvcourse").is_dir():
            for ruta in (candidato, candidato.parent):
                if str(ruta) not in sys.path:
                    sys.path.insert(0, str(ruta))
            return True
    return False


if not preparar_rutas():
    print("No encuentro `cvcourse`. En Colab, clona el repositorio primero:")
    print("    !git clone <url-del-repositorio>")
    print("    %cd legacyofInfest/computer-vision-course")

import numpy as np
import cv2

from cvcourse import engine_bridge, synthetic, viz

HAY_MOTOR = engine_bridge.hay_motor()
print(f"cvcourse listo. Motor disponible: {HAY_MOTOR}")
if not HAY_MOTOR:
    print("Sin motor: Canny etapa por etapa se explica sin la implementacion del repositorio.")

## 2. El concepto: la convolución

Un **kernel** es una matriz pequeña (3×3, 5×5) que codifica *qué mirar alrededor de cada píxel*. La **convolución** desliza ese kernel por toda la imagen y, en cada posición, multiplica y suma:

$$(f * k)[i, j] = \sum_{a,b} k[a, b] \, f[i-a, j-b]$$

es decir: el píxel nuevo vale la **media ponderada** de sus vecinos, con los pesos del kernel. De esa definición sale la propiedad más útil de la clase:

| Suma del kernel | Qué hace | Ejemplos |
|---|---|---|
| $\sum k = 1$ | conserva la energía: suaviza, afila | promedio, gaussiano, sharpen |
| $\sum k = 0$ | cancela zonas planas: **detecta cambios** | Laplaciano, Sobel |

Si el kernel suma 1 y todos los pesos son positivos, el resultado es un promedio y no puede tener más contraste que la entrada. Si suma 0, una zona plana da 0: solo responden los cambios. Comprueba ambas afirmaciones abajo.

In [ ]:
# Un trozo pequeño, un kernel de promedio, y la cuenta a mano:
trozo = np.array([[10, 20, 30], [40, 50, 60], [70, 80, 90]], dtype=float)
caja = np.ones((3, 3)) / 9.0

# El píxel central (50) pasa a ser la media de sus 9 vecinos.
print(f"central antes: {trozo[1, 1]:.1f}   media de los 9: {trozo.mean():.1f}")
print(f"suma del kernel de promedio : {caja.sum():.2f}   (conserva la energia)")


La convolución completa la hace `edge_detection.convolucionar` (NumPy puro, legible) o `cv2.filter2D` (OpenCV). Los dos deben dar lo mismo: la operación es la misma, el ejecutor es distinto.

In [ ]:
limpias, verdades = synthetic.lote_de_piezas(n=1, tamano=128, semilla=0)
base = limpias[0]

if HAY_MOTOR:
    from src.framework.processing import edge_detection

    con_numpy = edge_detection.convolucionar(base.astype(np.float32), caja)
else:
    def convolucionar_np(imagen, nucleo):
        r = nucleo.shape[0] // 2
        relleno = np.pad(imagen, r, mode="edge")
        salida = np.zeros_like(imagen, dtype=np.float32)
        for i in range(nucleo.shape[0]):
            for j in range(nucleo.shape[0]):
                salida += nucleo[i, j] * relleno[i:i + imagen.shape[0], j:j + imagen.shape[1]]
        return salida

    con_numpy = convolucionar_np(base.astype(np.float32), caja)

con_cv2 = cv2.filter2D(base, -1, caja)
print(f"misma operacion, dos ejecutores; diferencia media: {np.abs(con_cv2 - con_numpy).mean():.4f}")

viz.comparar(base, np.clip(con_cv2, 0, 255).astype(np.uint8),
             "original", "promedio 3x3", "La convolucion como operacion base");

## 3. Fundamento: tres modelos de ruido

El ruido es la diferencia entre lo que el sensor **vio** y lo que había delante. Tres modelos, tres causas, tres remedios:

| Modelo | Cómo se hace | De dónde viene | Se va con |
|---|---|---|---|
| **Gaussiano** | sumar $\mathcal{N}(0, \sigma)$ a cada píxel | ganancia del sensor, poca luz | promedio, gaussiano |
| **Sal y pimienta** | sustituir algunos píxeles por 0 o 255 | píxeles muertos, interferencias | **mediana** (y solo ella) |
| **Impulso/granular** | variar el valor por un factor | sensor barato, cuantización | gaussiano suave |

La lección de la Clase 1 decía que el **histograma no ve el ruido**. Mira qué pasa con las tres medidas de T2 —media, saturación, rango— cuando el ruido sube. Predice el resultado de cada fila antes de ejecutar.

In [ ]:
rng = np.random.default_rng(2)


def con_gaussiano(imagen, sigma):
    return np.clip(imagen.astype(float) + rng.normal(0, sigma, imagen.shape), 0, 255).astype(np.uint8)


def con_sal_y_pimienta(imagen, proporcion):
    sucia = imagen.copy()
    mascara = rng.random(imagen.shape) < proporcion
    sucia[mascara] = rng.integers(0, 2, mascara.sum()) * 255
    return sucia


print(f"{'imagen':16s} {'media':>7s} {'rango':>6s} {'sat%':>6s}   ruido?")
print("-" * 52)
ruidosas, etiquetas = [], []
for nombre, img in [
    ("limpia", base),
    ("gaussiana s=6", con_gaussiano(base, 6)),
    ("gaussiana s=20", con_gaussiano(base, 20)),
    ("sal y pimienta 2%", con_sal_y_pimienta(base, 0.02)),
    ("sal y pimienta 8%", con_sal_y_pimienta(base, 0.08)),
]:
    g = img.astype(float)
    print(
        f"{nombre:16s} {g.mean():>7.1f} {g.max() - g.min():>6.0f} "
        f"{(g >= 254).mean() * 100:>6.2f}   "
        f"{'no se ve' if (g >= 254).mean() * 100 == 0 else 'algo'}"
    )
    ruidosas.append(img)
    etiquetas.append(nombre)

viz.rejilla(ruidosas, etiquetas, columnas=5, titulo_general="El ruido que los histogramas no veian");

Las tres cifras casi no se mueven. **El ruido cambia dónde está cada valor, y el histograma tiró el dónde.** Lo que sí lo ve es un filtro espacial, que compara cada píxel con sus vecinos: un píxel aislado en 255 rodeado de 60 no puede ser la escena.

## 4. Ejercicio guiado: los tres filtros

Completa las tres funciones. Como la pieza sintética nos da la imagen **limpia**, el criterio es numérico: la RMSE (raíz del error cuadrático medio) contra la limpia. Un filtro que «se ve mejor» pero deja la RMSE alta no está haciendo su trabajo.

| Filtro | Idea | Frente al ruido |
|---|---|---|
| Promedio | media de la ventana | gaussiano |
| Gaussiano | media **ponderada**, central más importante | gaussiano |
| Mediana | el valor **central** ordenado, no la media | sal y pimienta |

La mediana no tiene fórmula: ordena los 9 vecinos y se queda con el 5.º. Por eso un 0 o un 255 aislado nunca gana la votación.

In [ ]:
def rmse(a, b):
    return float(np.sqrt(((a.astype(float) - b.astype(float)) ** 2).mean()))


def filtrar_promedio(imagen, k=3):
    nucleo = np.ones((k, k)) / k**2
    return cv2.filter2D(imagen, -1, nucleo)


def filtrar_gaussiano(imagen, sigma=1.2):
    return cv2.GaussianBlur(imagen, (5, 5), sigma)


def filtrar_mediana(imagen, k=3):
    return cv2.medianBlur(imagen, k)


sucia_g = con_gaussiano(base, 8)
sucia_s = con_sal_y_pimienta(base, 0.06)

print(f"{'filtro':18s} {'RMSE vs gaussiana':>18s} {'RMSE vs sal y pimienta':>22s}")
print("-" * 62)
print(f"{'sin filtrar':18s} {rmse(base, sucia_g):>18.1f} {rmse(base, sucia_s):>22.1f}")
for nombre, filtro in [
    ("promedio", filtrar_promedio),
    ("gaussiano", filtrar_gaussiano),
    ("mediana", filtrar_mediana),
]:
    print(
        f"{nombre:18s} {rmse(base, filtro(sucia_g)):>18.1f} "
        f"{rmse(base, filtro(sucia_s)):>22.1f}"
    )

viz.rejilla(
    [sucia_s, filtrar_promedio(sucia_s), filtrar_gaussiano(sucia_s), filtrar_mediana(sucia_s)],
    ["sal y pimienta 6%", "promedio", "gaussiano", "mediana"],
    columnas=4, titulo_general="Tres filtros contra el mismo ruido",
);

La columna de sal y pimienta es la que manda: el promedio y el gaussiano **esparcen** cada grano en una mancha gris —ese es su RMSE alto—, y solo la mediana los elimina. Cada ruido tiene su filtro, y aplicarlo a ciegas es peor que no filtrar.

**La RMSE baja, pero no a cero.** Los filtros suavizan también la pieza: un filtro perfecto contra ruido destruye un poco de señal. La tensión entre limpiar y conservar es el tema de toda la clase, y se resuelve con el kernel adecuado — nunca con «más fuerza».

## 5. Ejercicio grupal: diseñar un kernel propio

Un kernel se diseña con lápiz y se verifica con una cifra. Diseña uno que haga algo **útil y distinto** de los tres filtros: un realce de nitidez (sharpen), un detector de bordes direccional, un suavizado en cruz (solo horizontal+vertical), o lo que se te ocurra.

Antes de ejecutar, anota en el grupo: **¿cuánto suma tu kernel, y qué predice eso sobre su efecto?** Luego comprueba con el histograma y con una tarea.

Tres kernels de referencia, para partir de ahí:

| Nombre | Kernel | Suma | Efecto esperado |
|---|---|---|---|
| afilar (sharpen) | `[[0,-1,0],[-1,5,-1],[0,-1,0]]` | 1 | resalta diferencias locales |
| laplaciano | `[[0,1,0],[1,-4,1],[0,1,0]]` | 0 | bordes en las dos direcciones |
| emboss | `[[-2,-1,0],[-1,1,1],[0,1,2]]` | 0 | bordes diagonales con relieve |

In [ ]:
# TU kernel aquí. Condición: definirlo como una matriz numpy 3x3 o 5x5.
MI_KERNEL = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32)

print(f"suma del kernel: {MI_KERNEL.sum():.1f}  =>  "
      f"{'conserva energia (suaviza/afila)' if abs(MI_KERNEL.sum() - 1) < 1e-6 else 'detecta cambios (bordes)'}")

con_mi_kernel = cv2.filter2D(base, -1, MI_KERNEL)
print(f"RMSE del resultado, comparado con la limpia: {rmse(base, con_mi_kernel):.1f}")

viz.comparar(base, np.clip(con_mi_kernel, 0, 255).astype(np.uint8),
             "original", "con MI_KERNEL", "El efecto medido, no el efecto imaginado");

## 6. Aplicación industrial: sobre la superficie con defecto

La pieza con **grieta** es el caso que obliga a esto: la grieta es una línea fina y oscura, y su borde es el más tenue de los tres defectos. El pipeline completo de la clase lo monta `examples/class02_processing/industrial/superficie_con_defectos.py`; aquí se repite el paso decisivo: medir qué estrategia de filtrado deja que el borde de la grieta sobreviva a Canny.

La referencia se construye sin adivinar nada: Canny sobre la pieza **limpia** da el mapa de bordes que debería haber. Cualquier borde extra que Canny encuentre en la imagen sucia es ruido o mancha.

In [ ]:
limpia_grieta, _ = synthetic.pieza_individual(
    tamano=256, clase="NO_OK", defecto="grieta", ruido=0.0, semilla=3
)
# El generador devuelve gris (alto, ancho): mismo convenio para Canny.

rng2 = np.random.default_rng(3)
sucia = np.clip(limpia_grieta.astype(float) + rng2.normal(0, 3, limpia_grieta.shape), 0, 255).astype(np.uint8)
m = rng2.random(sucia.shape) < 0.02
sucia[m] = np.where(rng2.random(m.sum()) < 0.5, 0, 255)

referencia = cv2.Canny(limpia_grieta, 40, 120) > 0


def iou_bordes(a, b):
    union = float((a | b).sum())
    return float((a & b).sum()) / union if union else 1.0


print(f"{'estrategia':20s} {'IoU vs limpia':>14s} {'px borde':>9s}")
print("-" * 48)
for nombre, filtrada in [
    ("sin filtrar", sucia),
    ("gaussiano", cv2.GaussianBlur(sucia, (5, 5), 1.1)),
    ("mediana", cv2.medianBlur(sucia, 5)),
    ("gaussiano+mediana", cv2.medianBlur(cv2.GaussianBlur(sucia, (5, 5), 1.1), 5)),
]:
    bordes = cv2.Canny(filtrada, 40, 120) > 0
    print(f"{nombre:20s} {iou_bordes(bordes, referencia):>14.3f} {int(bordes.sum()):>9d}")

print("\nEl gaussiano antes de la mediana empeora: esparce la sal y pimienta.")
print("El orden de los filtros es parte del diseno, no un detalle.")

## 7. Bordes: la derivada discreta y Sobel

Un borde es un **cambio** de intensidad, y el cambio se mide con la derivada. En una imagen, la derivada horizontal se aproxima con el kernel $[-1, 0, 1]$: resta el píxel de la izquierda al de la derecha. Sobel añade pesos 2 en la fila central para suavizar y derivar a la vez:

$$G_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix} \qquad G_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix}$$

Y la **magnitud** del gradiente, que junta las dos direcciones:

$$|\nabla f| = \sqrt{G_x^2 + G_y^2}$$

**Pregunta de pizarra:** en $G_x$ la columna izquierda es negativa y la derecha positiva. ¿A qué bordes responde: a los verticales o a los horizontales? Ejecuta y compruébalo en la figura.

In [ ]:
gris = limpia_grieta.astype(np.float32)

if HAY_MOTOR:
    from src.framework.processing import edge_detection

    gx = edge_detection.convolucionar(gris, edge_detection.KERNEL_X)
    gy = edge_detection.convolucionar(gris, edge_detection.KERNEL_Y)
    magnitud = np.hypot(gx, gy)
else:
    gx = cv2.Sobel(gris, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gris, cv2.CV_32F, 0, 1, ksize=3)
    magnitud = cv2.magnitude(gx, gy)

print(f"magnitud: media {magnitud.mean():.1f}  maximo {magnitud.max():.1f}")

viz.rejilla(
    [np.abs(gx), np.abs(gy), np.clip(magnitud, 0, 255)],
    ["Gx (responde a bordes verticales)", "Gy (responde a bordes horizontales)", "magnitud"],
    columnas=3, titulo_general="Sobel: Gx, Gy y su combinacion",
);

## 8. Experimento: Canny, de la idea al algoritmo

Canny (1986) convirtió «bordes bonitos» en tres requisitos: **localización** (el borde donde está, no al lado), **respuesta única** (una línea, no una mancha) y **bajo ruido**. Las cuatro etapas son la traducción directa:

| Etapa | Requisito que cumple |
|---|---|
| 1. Suavizado gaussiano | el gradiente amplifica el ruido; se suaviza antes de derivar |
| 2. Gradiente + dirección | dónde cambia y hacia dónde |
| 3. Supresión no máxima | **respuesta única**: solo el píxel máximo en la dirección del gradiente |
| 4. Histéresis (doble umbral) | **bajo ruido**: los píxeles seguros propagan a sus vecinos candidatos |

Con el motor, cada etapa es una función con su nombre (`edge_detection.suavizar` → `gradiente` → `supresion_no_maxima` → `histeresis`). Sin motor, la etapa 1 y 2 se hacen con OpenCV y las 3 y 4 se discuten sobre sus números.

In [ ]:
UMBRAL_BAJO, UMBRAL_ALTO, SIGMA = 50.0, 150.0, 1.4

if HAY_MOTOR:
    suave = edge_detection.suavizar(gris, SIGMA)
    mag, ang = edge_detection.gradiente(suave)
    delgado = edge_detection.supresion_no_maxima(mag, ang)
    bordes = edge_detection.histeresis(delgado, UMBRAL_BAJO, UMBRAL_ALTO)
    etapas = [suave, mag, delgado, bordes]
    nombres = ["1. suavizado", "2. gradiente", "3. supresion", "4. histeresis"]
    print(f"tras el gradiente: {int((mag > UMBRAL_BAJO).sum()):>5d} px  |  tras la supresion: "
          f"{int((delgado > UMBRAL_BAJO).sum()):>5d} px  |  final: {int((bordes > 0).sum()):>5d} px")
    print("La supresion adelgaza; la histeresis deja solo lo que conecta con lo seguro.")
else:
    suave = cv2.GaussianBlur(gris, (5, 5), SIGMA)
    mag = cv2.magnitude(cv2.Sobel(suave, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(suave, cv2.CV_32F, 0, 1, ksize=3))
    bordes = cv2.Canny(sucia.astype(np.uint8), int(UMBRAL_BAJO), int(UMBRAL_ALTO))
    etapas = [suave, mag, bordes]
    nombres = ["1. suavizado", "2. gradiente", "3+4. Canny completo"]
    print("(sin motor: las etapas 3 y 4 estan dentro de cv2.Canny)")

etapas_viz = [gris.astype(np.uint8)] + [np.clip(e, 0, 255).astype(np.uint8) for e in etapas]
nombres_viz = ["gris"] + nombres
viz.rejilla(etapas_viz, nombres_viz, columnas=len(etapas_viz),
            titulo_general=f"Canny en sus etapas - umbrales {UMBRAL_BAJO:.0f}/{UMBRAL_ALTO:.0f}, sigma {SIGMA}");

## 9. Experimento grupal: romper Canny a propósito

Ya viste las cuatro etapas trabajando. Ahora **rómpelas tú**: 

* baja el **umbral alto** hasta que la histéresis acepte todo — el mapa de bordes se puebla de ruido;
* súbelo hasta arriba — desaparece hasta la grieta, porque nada pasa de «seguro»;
* sube el **bajo** hasta casi el alto — la propagación por vecindad se queda sin candidatos y el borde se desconecta en pedazos.

Para cada fila, cuenta píxeles de borde y **componentes conexas** (cuántos trozos): un buen Canny da pocos trozos; uno roto da miles o cero. Predice cada columna antes de ejecutar.

In [ ]:
from scipy import ndimage


def trozos(bordes):
    etiquetado, n = ndimage.label(bordes > 0)
    return n


print(f"{'umbral bajo':>11s} {'umbral alto':>11s} {'px de borde':>11s} {'trozos':>7s}  estado")
print("-" * 62)
for bajo, alto in [(50, 150), (10, 20), (10, 245), (140, 150), (50, 255), (50, 60)]:
    bordes = cv2.Canny(sucia, bajo, alto)
    n_px = int((bordes > 0).sum())
    n_trozos = trozos(bordes)
    if n_px == 0:
        estado = "vacio: nada pasa"
    elif n_trozos > 200:
        estado = "reventado de ruido"
    elif n_trozos < 5:
        estado = "contorno limpio"
    else:
        estado = "fragmentado"
    print(f"{bajo:>11d} {alto:>11d} {n_px:>11d} {n_trozos:>7d}  {estado}")

viz.rejilla(
    [cv2.Canny(sucia, b, a) for b, a in [(50, 150), (10, 20), (140, 150), (50, 60)]],
    [f"{b}-{a}" for b, a in [(50, 150), (10, 20), (140, 150), (50, 60)]],
    columnas=4, titulo_general="Los mismos umbrales, cuatro mundos");

**¿Por qué un solo umbral no basta?** Con uno, elegir el valor es elegir entre perder la grieta (umbral alto) o aceptar el ruido (umbral bajo). Canny usa dos por eso: la histéresis exige que un candidato **toque** a un píxel seguro para sobrevivir. El ruido, aislado, no toca nada y muere; la grieta, continua, se salva. La columna «trozos» es la que cuenta la historia: el ruido son miles de islas, el borde real es un continente.

Ese balance —dos umbrales que se deciden por conteo de trozos— es la primera vez en el curso que **el parámetro se elige por una medida**, no por «se ve bien».

## 10. Aplicación: el presupuesto de 16,6 ms

El videojuego dibuja a 60 fps: **16,6 ms por fotograma, para todo**. La línea de producción procesa 5 piezas/segundo: **200 ms por imagen**. La pregunta no es «¿qué filtro quiero?», es «¿qué filtro me cabe?».

La tabla de `rendimiento_convolucion.py` se reproduce aquí en pequeño, sobre la misma imagen. Los tiempos son de esta máquina; en otra cambian, y por eso todo el curso fija la imagen, el número de pasadas y la mediana: para que la comparación sea justa aunque los números no lo sean.

In [ ]:
import time

grande, _ = synthetic.pieza_individual(tamano=512, clase="OK", semilla=7)
nucleo = np.ones((3, 3), np.float32) / 9.0


def mediana_de_tiempos(fn, pasadas=7):
    fn()
    tiempos = []
    for _ in range(pasadas):
        t0 = time.perf_counter()
        fn()
        tiempos.append((time.perf_counter() - t0) * 1000)
    return float(np.median(tiempos))


candidatas = [("el propio del motor", edge_detection.convolucionar) if HAY_MOTOR else ("numpy a mano", convolucionar_np)]
candidatas += [
    ("scipy", lambda im, k: ndimage.convolve(im, k)),
    ("OpenCV filter2D", lambda im, k: cv2.filter2D(im, -1, k)),
]

print(f"{'implementacion':18s} {'3x3 (ms)':>9s} {'5x5 (ms)':>9s}  cabe a 60 fps?")
print("-" * 58)
for nombre, fn in candidatas:
    ms3 = mediana_de_tiempos(lambda: fn(grande, np.ones((3, 3), np.float32) / 9.0))
    ms5 = mediana_de_tiempos(lambda: fn(grande, np.ones((5, 5), np.float32) / 25.0))
    print(f"{nombre:18s} {ms3:>9.2f} {ms5:>9.2f}  {'SI' if ms3 < 16.6 else 'NO'}")

print("\nEl propio del motor cabe apenas en 3x3 y se sale en 5x5; OpenCV")
print("traga los dos. La misma matematica, ejecutores distintos: de ahi")
print("que el motor use OpenCV en el juego y la version lenta en el aula.")

## 11. Reto

Elige uno:

**A · Kernel separable.** El gaussiano 2D es el producto de dos 1D: convolucionar con $[\frac14, \frac12, \frac14] \times 2$ en las dos direcciones cuesta $O(2\cdot3)$ por píxel en vez de $O(9)$. Muéstralo: implementa la versión separable con `convolucionar` o con NumPy, compara la salida con `cv2.GaussianBlur` y mide el ahorro de tiempo.

**B · Romper la histéresis de otra manera.** Con un solo par de umbrales bueno, la grieta sale entera. Prueba a cortarla: pinta un segmento pequeño de la grieta con el color del fondo («si la grieta no toca el contorno, ¿sigue saliendo?»). La histéresis propaga desde lo seguro: explica con tus números qué hace falta para que un borde sobreviva.

**C · Veterano del aula.** Toma un sprite del motor (o una pieza sintética con defecto de **deformación**), aplícale el pipeline completo de la clase y decide, con cifras, cuánto filtrado aplicar antes de Canny para que el contorno exterior salga limpio. Compara con el IoU del §6.

In [ ]:
# Tu reto aquí.


## 12. Preguntas de análisis

Respóndelas en `analisis.md`. Cada una con una cifra o una figura detrás.

1. Un kernel que suma 0 deja las zonas planas en 0. ¿Por qué? ¿Qué pasa si el kernel suma 0 y el píxel es un máximo local de ruido?
2. En el §4, la RMSE contra la limpia no llega a cero con ningún filtro. ¿Qué información se destruye al filtrar, y cómo lo mediste? Distingue señal y ruido en tu respuesta.
3. El promedio deja la RMSE contra sal y pimienta alta. ¿Por qué la mediana gana donde el promedio pierde? Da la razón con la definición de cada uno, no con «se ve mejor».
4. En el §9, el par (50, 60) deja la grieta fragmentada. ¿Qué etapa de Canny produce esa fragmentación, y por qué subir el umbral bajo lo empeora en vez de arreglarlo?
5. La tabla de §10 depende de la máquina. ¿Qué partes del protocolo de medición la hacen *reproducible* pese a eso? ¿Qué afirmación sobre el codigo de la clase puedes hacer con tus números, y cual no?

## 13. Conclusiones

1. La **convolución** con un kernel es la operación base: promedios ponderados que suavizan (suma 1) o derivadas discretas que detectan cambios (suma 0).
2. Los filtros se eligen contra el **modelo de ruido**: gaussiano → gaussiano/promedio; sal y pimienta → mediana, y solo ella. Filtrar a ciegas es peor que no filtrar.
3. **Sobel** es la magnitud de dos derivadas; **Canny** convierte el gradiente en líneas: suaviza, deriva, adelgaza y deja pasar solo lo que conecta con lo seguro.
4. Los parámetros de Canny se eligen con una **medida** —píxeles, trozos, IoU— no por estética. Romper el algoritmo a propósito enseña qué hace cada umbral.
5. Todo esto cabe en 16,6 ms si las piezas pesadas las hace quien las hace bien (OpenCV, 0,5 ms la convolución 3×3) y la versión lenta se queda en el aula, donde se lee.

**Clase 3:** estos bordes limpios son la entrada de la segmentación: umbral, morfología, componentes conexas y las características geométricas que convierten píxeles en objetos medidos.

## Bibliografía

- Canny, J. *A Computational Approach to Edge Detection*. IEEE TPAMI, 1986.
- Gonzalez, R. C. y Woods, R. E. *Digital Image Processing*, 4.ª ed. Capítulo 3 (ecualización, suavizado) y capítulo 10 (§10.2, detección de bordes).
- Szeliski, R. *Computer Vision: Algorithms and Applications*, 2.ª ed. §3.2 (filtrado) y §4.2 (detección de bordes).
- Código del motor: `src/framework/processing/edge_detection.py` — Canny paso a paso en NumPy puro, y `filter_tools.py` para la versión OpenCV que usa el juego.